# **Elastic Pendulum**
## FEM Implementation using NGSolve
---------------------------------------

### **Theory**

The following implementation is a simple example of an elastic pendulum with wall impact using the Finite Element Method (FEM) with NGSolve.
Following references were used to implement the model:
- [Elastic Pendulum - NGS Tutorial 2024](https://docu.ngsolve.org/ngs24/tutorials/00_dynamics.html)
- [Contact Problems - NGS Docu Interactive Tutorial](https://docu.ngsolve.org/latest/i-tutorials/unit-6.2-contact/contact.html)
- [An Interactive Introduction to the Finite Element Method, Joachim Schöberl, TU Wien, ASC](https://jschoeberl.github.io/iFEM/intro.html)

#### **1. Modeling Elasticity**

##### **1.1. Kinematics**

|**Property**|**Definition**|
|---|---|
|Body in rest|$\Omega \subset \mathbb{R}^d$|
|Deformation function| $\phi : \Omega \rightarrow {\mathbb R}^3$|
|Displacement| $u(x) = \phi(x) - x$|
|Deformation Gradient| $F = \nabla \phi$|
|Cauchy-Green Strain Tensor| $C = F^T F$|
|Rigid Body Motion| $\phi(x) = a + Qx$, $a \in \mathbb{R}^3$, $Q$ is a rotation matrix |
|Green's Deformation Tensor| $E = \frac{1}{2}(C - I)=\frac{1}{2}\big( \nabla u + \nabla u^T + \nabla u^T \nabla u \big)$|

The **Green Strain Tensor** $C$ measures the relative squared change of length in the point $x$ into the direction $\Delta x$:

$$
\frac{\| \phi(x+\Delta x) - \phi(x)\|^2}{\| x + \Delta x - x \|^2} = \frac{\| F(x) \Delta x + O(\| \Delta x \|^2) \|^2} {\| \Delta x \|^2} = \frac{ \Delta x^T F^T F \Delta x} { \| \Delta x\|^2 } + O(\|\Delta x \|)
$$

If $C=I$, then the body is not deformed and $\phi$ is a rigid body motion.


##### **1.2. Elastic Material Laws**

- When work is applied to a body, it deforms and stores deformation energy. 
- An elastic body returns the work when the external forces are removed.

**Hyperelastic materials**:

Constitutive law which expresses the deformation energy point-wise by the Cauchy-Green strain tensor:
$$
E_{def} = \int_\Omega W(C(u)) \, dx
$$

- Energy density function $W$ may depend on the position $x$ when the material of the body is inhomogeneous (i.e. $W = W(x, C(u))$).

**Isotropic constritution equation**:

- The material properties are the same in all direction.
- The deformation energy is independent of the rotation of the body before deformation.
- Thus, $W$ is a function the (real and positive) eigenvalues $\lambda_i$ of $E$
- **Characteristic polynomial**: $$\det (\lambda I - E) = \lambda^3 - I_1(E) \lambda^2 + I_2(E) \lambda - I_3(E)$$ with the invariants
\begin{align*}
        I_1(E) & =  \operatorname{tr} (E) = \lambda_1 + \lambda_2 + \lambda_3 \\
        I_2(E) & =  \frac{1}{2} [ (\operatorname{tr} E)^2 - \operatorname{tr} (E^2) ] = \lambda_1 \lambda_2 + \lambda_1 \lambda_3 + \lambda_2 \lambda_3 \\
        I_3(E) & =  \det (E) = \lambda_1 \lambda_2 \lambda_3
\end{align*}

- **Rivelin-Ericksen-Theorem** for isotropic materials: $$W(E) = W( \operatorname{tr} (E), \operatorname{tr}(E^2), \det (E) )$$
- Assuming that $W(E=0) = 0$ is a local minimum and $W$ is smooth, we can expand $W$ in a Taylor series around $E=0$:
$$
W(E) = \frac{1}{2} W_{,11} \operatorname{tr} (E)^2 + W_{,2} \operatorname{tr} (E^2) + O(\| E \|^3)
$$

- Dropping higher order terms and Lamé parameters $\lambda := W_{,11}$ (resistance to volumetric deformation) and $\mu := W_{,2}$ (resistance to shear deformation) we obtain the second order energy density (called *Hooke's Law*):
$$
W(E) = \frac{\lambda}{2} \operatorname{tr} (E)^2 + \mu E:E,
$$ 
- It is similar to an elastic spring: The stored energy is $\frac{1}{2} k E^2$, with the spring constant $k$ and elongation $E$.

##### **1.3. Variational Formulation and Equilibrium**

- Let $V$ be a function space of feasible displacements and $\Gamma_D$ the Dirichlet boundary where the displacement is prescribed

- **Total energy = Deformation energy + Potential of external forces**:
$$
J(u) = \int_\Omega W(C(u)) \, dx - \int_\Omega f \cdot u \, dx - \int_{\Gamma_N} g \cdot u \, ds 
$$
with 
  - $f$: volume force density
  - $g$: force density (traction) on the Neumann boundary $\Gamma_N$
  - Arguments of $f$ and $g$ are the position $x$ in the reference configuration.

**Procedure**:

1. We search for the displacement $u \in V$ that minimizes the total energy $J(u)$:
    $$
    \min_{u \in V} J(u)
    $$

2. We compute the directional derivative in the direction $v$ (such that $u+\varepsilon v$ is feasible as well): 

$$
\left< J^\prime(u), v\right> = \int_\Omega \frac{d W}{d C}(C(u)) \frac{d C}{d u}(u) v \, dx - \int_\Omega f v  - \int_{\Gamma_N} gv 
$$

3. We define the *$2^{nd}$ Piola Kirchhoff stress tensor* as ($\Sigma$ is a symmetric matrix)

$$
\Sigma = 2 \frac{dW}{dC}.
$$ 

4. From $C = F^T F = (I + \nabla u)^T (I + \nabla u)$ there follows 

$$
\frac{d C}{d u} v = (I + \nabla u)^T \nabla v + \nabla v^T (I + \nabla u),
$$

5. Further, we obtain
\begin{align*}
\left< J^\prime(u), v\right> & = \int_\Omega \Sigma : F^T \nabla v  - \int_\Omega f v - \int_{\Gamma_N} g v  \\
 & = \int_\Omega F \Sigma : \nabla v - \int_\Omega f v - \int_{\Gamma_N} g v 
\end{align*}

6. This term must vanish for all feasible directions $v$:

$$
 \int_\Omega F \Sigma : \nabla v  = \int_\Omega f v - \int_{\Gamma_N} g v \qquad \forall \, v  \in V(\Omega)
$$

7. From integration by parts we obtain the equilibrium of forces:

\begin{align*}
-\operatorname{div} F \Sigma & = f \qquad \text{ in } \Omega \\
F \Sigma n  & = g \qquad \text{ on } \Gamma_N
\end{align*}

8. We obtain the *$1^{st}$ Piola-Kirchhoff stress tensor* (in general $P$ is not symmetric)
$$
P = F \Sigma 
$$


9. Transform the integrals to the deformed configuration $\phi(\Omega)$, set $v = \tilde v \circ \phi$ with a test-function $\tilde v$ from a proper space
$V (\phi(\Omega))$. The composition $\tilde v \circ \phi$ means that we evaluate $\tilde v$  at the deformed position $\phi(x)$ instead of $x$. From the chain-rule there follows $\nabla v = \nabla \tilde v \, F$. Notation: $J = \det F$:

$$
\int_{\phi(\Omega)} J^{-1} F \Sigma : \nabla \tilde v F \, dx = \int_{\phi(\Omega)} J^{-1} f \tilde v + \int_{\phi(\Gamma_N)} \tilde g \tilde v \qquad \forall \tilde v \in V(\phi(\Omega))
$$

$$
\int_{\phi(\Omega)} J^{-1} F \Sigma F^T : \nabla \tilde v \, dx = \int_{\phi(\Omega)} J^{-1} f \tilde v + \int_{\phi(\Gamma_N)} \tilde g \tilde v  \qquad \forall \tilde v \in V(\phi(\Omega))
$$

10. One obtains $\tilde g = \frac{ds}{\tilde{ds}} g$ from the transformation of surface measures.

11. *Cauchy stress tensor*: It is symmetric and satisfies equilibrium of forces on the deformed configuration:

$$
\sigma :=  J^{-1} F \Sigma F^T
$$

\begin{align*}
-\operatorname{div} \sigma & = J^{-1} f \qquad \text{ in } \phi(\Omega) \\
\sigma n & = \tilde g \quad \quad \qquad \text{on }  \phi(\Gamma_N)
\end{align*}

#### **2. Solving Nonlinear Elasticity Variational Problems**

We consider minimization problems of the form

$$\text{find } u \in V \text{ s.t. } E(u) \leq E(v) \quad \forall~  v \in V.$$

- We are solving this problem using Newton's method.
- We are using the `Variation` integrator of `NGSolve` and formulate the problem through a symbolic description of an energy functional.
- Let $E(u)$ be the energy that is to be minimized for the unknown state $u$.
- A necessary optimality condition is that the derivative at the minimizer $u$ in all directions $v$ vanishes, i.e. 
$$
  \delta E(u) (v) = 0 \quad \forall v \in V
$$

- We assume for our pendulum a Neo-Hookean hyperelastic material model.
- The energy density function is given as
$$
  E(v) := \int_{\Omega} \frac{\mu}{2} ( \operatorname{tr}(F^T F-I)+\frac{2 \mu}{\lambda} \operatorname{det}(F^T F)^{\frac{\lambda}{2\mu}} - 1) ~~ dx
$$

#### **3. Solving Dynamic Contact Problems**

- Dynamic contact combines for our example nonlinear elasticity with contact constraints in a time-stepping scheme.
- The approach uses a penalty method to enforce the contact conditions and to handle large deformations through the Neo-Hookean material models.

1. **Geometry and Mesh**:
- Contact boundaries must be explicitly defined
- Self-contact is supported (same boundary can contact itself)

2. **Contact Gap Function**:

- Measures the signed distance between the two contact surface
- Negative values indicate penetration

``` python
cf = (X + u-uold - (X.Other() + u.Other() - uold.Other())) * (-specialcf.normal(2).Other())
```
Where 
- `X`: reference position
- `u`: current displacement
- `uold`: displacement from previous time step
- `Other()`: values from the other side of the contact boundary
- `specialcf.normal(2)`: normal vector on the contact boundary (pointing outward from the element)
- `cf`: gap function, negative values indicate penetration

3. **Contact Energy**:

``` python
contact.AddEnergy(IfPos(cf, 1e9*cf*cf, 0), deformed=True)
```
- Penalty method: large stiffness ($1e9$) when surfaces penetrate (cf < 0)
- `IfPos(cf, 1e9*cf*cf, 0)`: adds energy only when `cf` is positive (penetration)
- `deformed=True`: evaluates the energy in the deformed configuration

#### **4. Newmark time-stepping**

References:
- [Newmark-beta method - Wikipedia](https://en.wikipedia.org/wiki/Newmark-beta_method)
- [NGSolve Documentation - Newmark Method](https://docu.ngsolve.org/ngs24/SaS/dynamics_newmark_gen_alpha.html)

The Newmark method is an implicit time-integration scheme for solving second-order differential equations (structural dynamics problems).

1. **Mathematical Foundation**
- The scheme solves the dynamic equilibrium equations by approximating the displacement, velocity, and acceleration at each time step.
$$
M \ddot{u} + C \dot{u} + f^{int} u = f^{ext}
$$
- where:
    - $M$: mass matrix
    - $C$: damping matrix 
    - $f^{int}$: internal force vector
    - $f^{ext}$: external force vector

- New acceleration is obtained from the elasticity operator $K$:
$$
a^{n+1} = f - K(u^{n+1})
$$
- where:
    - Displacement $u^{n+1}$ and the acceleration $a^{n+1}$ at the new time step are unknowns
    - The velocity has to be determined via the time stepping scheme (see below)

2. **Newmark Scheme**

- Trapezoidal rule (average acceleration method) is used to approximate the velocity and acceleration.
- Implicit method: requires solving a nonlinear system at each time step (e.g., using Newton's method).

Key equations of the Newmark method are:

\begin{align}
\frac{u^{n+1}-u^n}{\tau} &= \frac{v^n+v^{n+1}}{2} \\
\frac{v^{n+1}-v^n}{\tau} &= \frac{a^n+a^{n+1}}{2}
\end{align}

They can be rearranged to express $v^{n+1}$ and $a^{n+1}$ in terms of $u^{n+1}$:

\begin{align}
v^{n+1} &= \frac{2}{\tau}(u^{n+1} - u^n) - v^n \\
a^{n+1} &= \frac{2}{\tau}(v^{n+1} - v^n) - a^n
\end{align}

3. **Procedure**

    1. **Prediction:** Start with known values at time step $n$ ($u^n$, $v^n$, $a^n$)
    2. **Implicit Solution:** Solve nonlinear system for $u^{n+1}$ using Newton's method.
    3. **Correction:** Update $v^{n+1}$ and $a^{n+1}$ using the equations above.
    4. **Advance:** Move to the next time step and repeat.

#### **5. Constraint Implementation - Pendulum Pivot**

```python
bfa += (InnerProduct(u, p) + InnerProduct(v, q)) * ds('top')
```

- Lagrange multipliers `p` and `q` are introduced to enforce the constraints at the pendulum pivot.
- Mean value constraint: Controls the average displacement of pivot point.
- Allows the pendulum to swing freely while keeping the pivot fixed.

#### **6. Variational Form - Weak Formulation**

- **Principle of Virtual Work**: The weak form of the equilibrium equations is derived from the principle of virtual work, which states that the work done by internal forces equals the work done by external forces for any virtual displacement
$$
\delta W_{\text{internal}} = \delta W_{\text{external}}
$$

- `Variation(NeoHooke(C(u))*dx).Compile()`: Computes the variation of the Neo-Hookean energy density integrated over the domain. Corresponds to the internal elastic work from material nonlinearity.
- `acc_new*v*dx`: Represents the inertial forces due to acceleration.
- `-force*v*dx`: Represents the work done by external forces (e.g., gravity).


#### **7. Key Points**

1. **Large Deformations**: \
The Neo-Hookean material model captures large deformations and nonlinear elasticity.

2. **Contact Handling**: \
The penalty method effectively enforces contact constraints, preventing interpenetration.

3. **Time Integration**: \
The Newmark method provides a stable and accurate time-stepping scheme for dynamic simulations.

4. **Constraint Enforcement**: \
Lagrange multipliers are used to enforce constraints at the pendulum pivot, allowing for realistic motion.

5. **Variational Formulation**: \
The weak form derived from the principle of virtual work ensures that the internal and external forces are balanced for any virtual displacement.

#### **8. Questions**

1. Is it also possible to implement a rigid linear elastic material model in NGSolve for the pendulum?

2. Are there some adaptions required for handling proper gravity forces in the 2D case?

3. How to set the initial conditions for velocity and acceleration properly? Currently only correct if the initial angular position is considered during the creation of the geometry, not when applying to the grid function.

4. How to calculate the total energy of the system (kinetic + potential + elastic) at each time step to monitor energy conservation?

5. How to animate the stress distrubition while also showin the deformed shape of the pendulum?

6. How to verify the contact implementation? Animation does not look physically correct.

8. How to implement different material properties for pendulum and wall?

9. What are appropriate and feasible extensions of the model? (e.g., damping, external forces, more complex geometries, 3D simulation)

--------------------------------------

#### **Problem Description**

- Elastic pendulum with large deformations and wall contact using Neo-Hookean material model.
- Currently pendulum and wall using the same material properties.
- Pendulum is fixed at the top and swings under the influence of gravity, initial velocity, and initial angular acceleration (if defined) around the pivot joint (z-axis).
- The wall is fixed at the top and bottom edges.
- A contact condition is defined between the pendulum head edge and the $X+$ edge of the wall.


|Pendulum Geometry|Boundary and Initial Condition|
|-----------------|------------------------------|
|![](images/img_dimensions.png)|![](images/img_setup.png)|

In [2]:
import numpy as np
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *
from ngsolve.solvers import Newton
import ipywidgets as widgets

### Configuration

In [3]:
from calendar import day_abbr
from dataclasses import dataclass

@dataclass
class GeometryParameters:
    # Pendulum geometry
    r_rod:    float = 0.05
    r_hole:   float = 0.1
    r_head:   float = 0.2
    l_center: float = 0.8
    
    # Wall geometry
    q_wall_deg: float = 0
    wall_len_x: float = 0.15
    wall_len_y: float = 0.8
    wall_len_z: float = 0.1
    
@dataclass
class MaterialParameters:
    E:  float = 2100      # Young's modulus
    nu: float = 0.2       # Poisson ratio
    rho: float = 1        # density
    
@dataclass
class MeshParameters:
    max_element_size: float = 0.05
    mesh_order: int = 2
    curved_elements: bool = True
    refinement_levels: int = 0
    
@dataclass
class InitialConditionParameters:
    angular_position_deg: float = 45
    angular_velocity: float = 0
    angular_acceleration: float = 0
    
@dataclass
class SimulationParameters:
    t_start: float = 0       # start time
    tau: float = 0.025       # time step size
    t_end: float = 5         # end time    
    
@dataclass
class AnimationParameters:
    interval: int = 10       # time steps between frames
    speed: float = 2         # animation speed multiplier

### Pendulum Class

In [21]:
from turtle import pen


class SimpleFEMPendulum:
    def __init__(self):
        # Parameters
        self.geom_params = GeometryParameters()
        self.mat_params = MaterialParameters()
        self.mesh_params = MeshParameters()
        self.init_params = InitialConditionParameters()
        self.sim_params = SimulationParameters()
        self.anim_params = AnimationParameters()
        
        # Internal states
        self._mesh = None
        self._fes = None
        self._contact = None
        self._material_law = None
        self._simulation_results = []
        
        # Grid Functions
        self._gf_u = None
        self._gf_v = None
        self._gf_a = None
        self._gf_uold = None
        self._gf_vold = None
        
        # Results history
        self._gf_u_history = None
        self._gf_v_history = None
        
        self._setup_material_law()

        pass
    
    def _setup_material_law(self):
        self.E = self.mat_params.E
        self.nu = self.mat_params.nu
        self.rho = self.mat_params.rho
        
        # Lamé parameters
        self.mu = self.E / 2 / (1+self.nu)
        self.lam = self.E * self.nu / ((1+self.nu)*(1-2*self.nu))
        
        self._deformation_tensor = self.C
        self._material_law = self.NeoHooke
        pass
    
    def create_geometry(self):
        gp = self.geom_params

        bar = MoveTo(-gp.r_rod,0).Rectangle(2*gp.r_rod, gp.l_center).Face()
        bar.edges.Min(Y).name="rotation"
        bar.faces.name="bar"
        bar.faces.maxh=0.05

        hole = Circle((0, gp.l_center), gp.r_hole).Face()
        hole.faces.name="hole"

        circ = Circle((0, gp.l_center), gp.r_head).Face()
        circ.faces.name="circ"
        circ.faces.maxh=0.033
        circ.edges.name="contact_head"
        
        head = circ - hole
        pendulum = head + bar - hole

        #pendulum = Glue([bar - (circ), circ - hole])
        pendulum = pendulum.Rotate(Axis((0.0, 0, 0), (0, 0, 1)), 180)
        #pendulum = pendulum.Rotate(Axis((0.0, 0, 0), (0, 0, 1)), self.init_params.angular_position_deg)
        pendulum.name = "pendulum"

        # Wall
        wall_pos_x = -gp.r_head - gp.wall_len_x
        wall_pos_y = -gp.l_center - gp.wall_len_y/2
        wall = MoveTo(wall_pos_x, wall_pos_y).Rectangle(gp.wall_len_x, gp.wall_len_y).Face()
        wall.faces.maxh = 0.033
        wall.edges.Max(X).name = "contact_wall"
        wall.edges.Max(Y).name = "fix"
        wall.edges.Min(Y).name = "fix"

        self._geo = Compound([pendulum, wall])
    
    def create_mesh(self):
        self.create_geometry()
        
        mp = self.mesh_params
        
        geo = self._geo
        mesh = Mesh(OCCGeometry(geo, dim=2).GenerateMesh(maxh=mp.max_element_size))
        if mp.curved_elements:
            mesh.Curve(4)
        for i in range(mp.refinement_levels):
            # Mesh refinement
            mesh.Refine()
        
        self._mesh = mesh
        
    def _initialize_fe_spaces(self):
        # Create H1 vector space for 3D quantities (displacement, velocity, acceleration)
        self._V = VectorH1(self._mesh, order=3, dirichlet="fix")
        
        # Create NumberSpace for Lagrange multipliers (rotation constraint)
        self._Q = NumberSpace(self._mesh, definedon=self._mesh.Boundaries('rotation'))
        
        # Mixed FE space
        self._fes = self._V * self._Q**2
        (self._u, self._q), (self._v, self._p) = self._fes.TnT()
        
        # Scalar H1 space for stress
        self._S = H1(self._mesh, order=3)
        
        self._setup_material_law()      
        
        pass
    
    def _initialize_grid_functions(self):
        # Initialize grid functions
        self._gf_u = GridFunction(self._fes)  # Current state
        self._gf_v = GridFunction(self._fes)  # Velocity
        self._gf_a = GridFunction(self._fes)  # Acceleration
        
        self._gf_uold = GridFunction(self._fes)  # Previous displacement
        self._gf_vold = GridFunction(self._fes)  # Previous velocity
        self._gfaold = GridFunction(self._fes)  # Previous acceleration
        
        self._gf_stress = GridFunction(self._S) # Scalar grid function for stress                  
        
        # Time series storage
        self._gf_u_history = GridFunction(self._V, multidim=0)
        self._gf_v_history = GridFunction(self._V, multidim=0)
        self._gf_stress_history = GridFunction(H1(self._mesh, order=3), multidim=0)
    
    def _set_initial_conditions(self):
        # Convert to radians
        icp = self.init_params
        #theta = 0
        theta = np.deg2rad(icp.angular_position_deg)   # initial angular position
        omega = icp.angular_velocity                   # initial angular velocity
        alpha = icp.angular_acceleration               # initial angular acceleration
        
        c, s = np.cos(theta), np.sin(theta)
        
        # Rotation center (could be made configurable)
        cx = cy = 0.0
        
        if self._mesh.dim == 2:
            u_rot = CF(((c - 1.0) * (x - cx) - s * (y - cy),
                        s * (x - cx) + (c - 1.0) * (y - cy)))
            v_rot = CF((-omega * (y - cy),
                        omega * (x - cx)))
            a_rot = CF((-alpha * (y - cy),
                        alpha * (x - cx)))
        else:
            u_rot = CF(((c - 1.0) * (x - cx) - s * (y - cy),
                        s * (x - cx) + (c - 1.0) * (y - cy),
                        0))
            v_rot = CF((-omega * (y - cy),
                        omega * (x - cx),
                        0))
            a_rot = CF((-alpha * (y - cy),
                        alpha * (x - cx),
                        0))
        
        # Set initial conditions
        self._gf_u.components[0].Set(u_rot, definedon=self._mesh.Materials("pendulum"))
        self._gf_v.components[0].Set(v_rot, definedon=self._mesh.Materials("pendulum"))
        self._gf_a.components[0].Set(a_rot, definedon=self._mesh.Materials("pendulum"))

        # Copy to "old" variables
        self._gf_uold.vec[:] = self._gf_u.vec
        self._gf_vold.vec[:] = self._gf_v.vec
        self._gfaold.vec[:] = self._gf_a.vec
        pass
    
    def _initialize_contact(self):
        k_n = 1e9
        
        # Define Contact boundary
        slave = self._mesh.Boundaries("contact_head")
        master = self._mesh.Boundaries("contact_wall")
        self._contact = ContactBoundary(slave, master)
        
        # self._contact = ContactBoundary(self._mesh.Boundaries("contact_head|contact_wall"),
        #                                 self._mesh.Boundaries("contact_head|contact_wall"))
        # Setup X (= current position in space) and uold (= previous displacement)
        # X = CoefficientFunction((x,y))
        # self._uold = self._gf_uold.components[0]
        
        X = CoefficientFunction((x,y))
        nM = specialcf.normal(self._mesh.dim).Other()
        self._uold = self._gf_uold.components[0]
        
        # Define contact condition function (gap function)
        # Penetration is detected if cf < 0
        #gap_function = (X + self._u-self._uold - (X.Other() + self._u.Other() - self._uold.Other())) * (-specialcf.normal(2).Other())
        
        # Add contact energy to the contact object
        #self._contact.AddEnergy(IfPos(gap_function, penalty_stiffness*gap_function*gap_function, 0), deformed=True)
        
        gap = ((X + self._u) - (X.Other() + self._u.Other())) * nM
        
        # Add contact energy to the contact object
        self._contact.AddEnergy(IfPos(-gap, k_n*gap*gap, 0), deformed=True)
        pass
    
    def _setup_bilinear_form(self):
        # Bilinear form
        self._bfa = BilinearForm(self._fes)
        
        #self._bfa += Variation(self.NeoHooke(self.C(self._u))*dx).Compile() 
        self._bfa += Variation(self._material_law(self._deformation_tensor(self._u))*dx).Compile()
        
        # Rotation constraint
        self._bfa += (InnerProduct(self._u, self._p) + InnerProduct(self._v, self._q)) * ds('rotation')
        
        self.tau = self.sim_params.tau  # time step size
        self.tend = self.sim_params.t_end
        force = CF((0, -1))  # gravity force

        vel_new = 2/self.tau * (self._u-self._gf_uold.components[0]) - self._gf_vold.components[0]
        acc_new = 2/self.tau * (vel_new-self._gf_vold.components[0]) - self._gfaold.components[0]

        # need to add to the bilinear form since it depends on the current valurs of the GridFunctions
        self._bfa += acc_new*self._v*dx
        self._bfa += -force*self._v*dx
        
    def initialize(self):
        self._initialize_fe_spaces()
        self._initialize_grid_functions()
        self._set_initial_conditions()
        self._initialize_contact()
        self._setup_bilinear_form()
        pass
    
    def simulate(self):
        self._gf_u.vec[:] = 0
        self._gf_u_history.AddMultiDimComponent(self._gf_u.components[0].vec)
        #scene = Draw(gfu.components[0], mesh, "deformation", deformation=True)  
        t = self.sim_params.t_start
        i = 1
        with TaskManager():
            while t < self.tend:
                i += 1
                t += self.tau

                self._contact.Update(self._gf_u.components[0], self._bfa)

                Newton(a=self._bfa, u=self._gf_u, printing=False, inverse="sparsecholesky")

                self._gf_v.vec[:] = 2/self.tau * (self._gf_u.vec-self._gf_uold.vec) - self._gf_vold.vec
                self._gf_a.vec[:] = 2/self.tau * (self._gf_v.vec-self._gf_vold.vec) - self._gfaold.vec

                self._gf_uold.vec[:] = self._gf_u.vec
                self._gf_vold.vec[:] = self._gf_v.vec
                self._gfaold.vec[:] = self._gf_a.vec
                
                if i % self.anim_params.interval == 0:
                    # Compute stress from current displacement
                    C_ = self.C(self._gf_u.components[0]).MakeVariable()
                    sigma = self.NeoHooke(C_).Diff(C_)
                    self._gf_stress.Set(sigma[0,0])
                    
                    # Add 
                    self._gf_u_history.AddMultiDimComponent(self._gf_u.components[0].vec)
                    self._gf_v_history.AddMultiDimComponent(self._gf_v.components[0].vec)
                    self._gf_stress_history.AddMultiDimComponent(self._gf_stress.vec)
                
    def visualize(self, mesh=True, u=True, v=True, a=True):
        if mesh:
            # Mesh Geometry
            tw_geometry = widgets.Text(value="Mesh Geometry")
            display(tw_geometry)
            Draw(self._mesh, "mesh")
        if u:
            # Displacement
            tw_displacement = widgets.Text(value="Displacement")
            display(tw_displacement)
            Draw(self._gf_u.components[0], deformation=True)
        if v:
            # Velocity
            tw_velocity = widgets.Text(value="Velocity")
            display(tw_velocity)
            Draw(self._gf_v.components[0], deformation=self._gf_u.components[0], vectors=True)
        if a:
            # Acceleration
            tw_acceleration = widgets.Text(value="Acceleration")
            display(tw_acceleration)
            Draw(self._gf_a.components[0], deformation=self._gf_u.components[0], vectors=True)
        
    def animate_u(self):
        settings = {"Multidim": {
                    "speed" : self.anim_params.speed
                }};
        
        tw_u = widgets.Text(value="Displacement History Animation")
        display(tw_u)

        Draw(self._gf_u_history,
             self._mesh,
             interpolate_multidim=True,
             deformation=True,
             animate=True,
             autoscale = False,
             min = 0, max = 1,
             settings = settings);
        
    def animate_stress(self):
        settings = {"Multidim": {
                    "speed" : self.anim_params.speed
                }};
        
        tw_stress = widgets.Text(value="Stress History Animation")
        display(tw_stress)
        
        # Animate stress history and deformation history in one scene
        Draw(self._gf_stress_history,
             self._mesh,
             interpolate_multidim=True,
             animate=True,
             settings = settings);
        
    def C(self, u):
        F = Id(u.dim) + Grad(u)
        return F.trans * F
        
    def NeoHooke (self, C):
        return 0.5*self.mu*(Trace(C-Id(self._u.dim)) + 2*self.mu/self.lam*Det(C)**(-self.lam/2/self.mu)-1)

### Simulation

In [22]:
myPendulum = SimpleFEMPendulum()
myPendulum.create_mesh()
myPendulum.initialize()
myPendulum.visualize()

Text(value='Mesh Geometry')

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

Text(value='Displacement')

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

Text(value='Velocity')

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

Text(value='Acceleration')

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [23]:
myPendulum.simulate()

In [24]:
myPendulum.animate_u()

Text(value='Displacement History Animation')

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Multidim': {'speed': 2}}, '…

In [8]:
myPendulum.animate_stress()

Text(value='Stress History Animation')

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Multidim': {'speed': 2}}, '…